In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import ffmpeg

audio_paths = []
for video_path in Path("../tests/videos").glob("*.mp4"):
    audio_path = video_path.with_suffix(".wav")
    ffmpeg.input(str(video_path)).output(str(audio_path)).run(quiet=False, overwrite_output=True)
    audio_paths.append(audio_path)

In [4]:
from src.audio import extract_music_features

features = extract_music_features(audio_paths[0])

In [ ]:
import aubio
import numpy as np

def aubio_running_global_generator(audio_path, win_size=1024, hop_size=512, sample_rate=0):
    """
    一个自适应的流式生成器，既返回当前帧的局部特征，
    也返回【截至目前已处理音频】的全局渐进特征。
    """
    # 1. 初始化音频源与基础转换器
    audio_src = aubio.source(audio_path, sample_rate, hop_size)
    sr = audio_src.samplerate
    pv = aubio.pvoc(win_size, hop_size)

    # 2. 初始化特征检测器
    centroid_detector = aubio.specdesc("centroid", win_size)
    flux_detector = aubio.specdesc("specflux", win_size)
    onset_detector = aubio.specdesc("default", win_size)
    
    # 3. 初始化专门用于实时 BPM 追踪的 tempo 对象
    tempo_detector = aubio.tempo("default", win_size, hop_size, sr)

    # --------------------------------------------------
    # 状态计数器：用于在不存储历史数据的前提下，递推计算全局指标
    # --------------------------------------------------
    frame_count = 0
    
    # 用于渐进式动态范围 (Running Dynamic Range)
    running_max_rms = -float('inf')
    running_min_rms = float('inf')

    total_frames = 0
    
    try:
        while True:
            samples, read = audio_src()
            current_time = total_frames / float(sr)
            frame_count += 1
            
            # --- A. 基础局部特征提取 ---
            rms_val = aubio.level_lin(samples)
            ffted = pv(samples)
            
            centroid_val = centroid_detector(ffted)[0]
            flux_val = flux_detector(ffted)[0]
            onset_env_val = onset_detector(ffted)[0]
            
            # 同时把样本喂给 tempo 引擎，让它在底层默默更新 BPM 状态
            tempo_detector(samples)
            
            # --------------------------------------------------
            # B. 核心：递推更新全局特征 (Running Globals)
            # --------------------------------------------------
            
            # 1. 实时更新平均频谱质心 (Running Mean)
            # 公式: New_Mean = Old_Mean + (New_Value - Old_Mean) / N
            # 这种做法免去了存储一个巨大 list 的麻烦
            if frame_count == 1:
                running_avg_centroid = centroid_val
            else:
                running_avg_centroid += (centroid_val - running_avg_centroid) / frame_count
                
            # 2. 实时更新动态范围 (Running Dynamic Range)
            # 只考虑有效声音（过滤极度静音帧，避免环境噪音干扰边界）
            if rms_val > 0.0001: 
                if rms_val > running_max_rms: running_max_rms = rms_val
                if rms_val < running_min_rms: running_min_rms = rms_val
            
            # 计算截至目前的动态范围极限差
            if running_max_rms != -float('inf') and running_min_rms != float('inf'):
                running_dynamic_range = running_max_rms - running_min_rms
            else:
                running_dynamic_range = 0.0

            # 3. 实时从底层引擎获取最新预估的全局 BPM
            running_bpm = tempo_detector.get_bpm()

            # --------------------------------------------------
            # C. 统一产出
            # --------------------------------------------------
            yield {
                "time": current_time,
                "is_last_frame": (read < hop_size),
                # === 局部数据 (仅代表当前帧) ===
                "local": {
                    "rms": rms_val,
                    "spectral_centroid": centroid_val,
                    "spectral_flux": flux_val,
                    "onset_envelope": onset_env_val
                },
                # === 渐进全局数据 (代表从 0 秒到当前时间的整体特征) ===
                "running_global": {
                    "average_spectral_centroid": running_avg_centroid,
                    "overall_brightness_hz": running_avg_centroid, # 质心均值即代表整体明亮度
                    "dynamic_range": running_dynamic_range,
                    "estimated_bpm": running_bpm
                }
            }
            
            total_frames += read
            if read < hop_size:
                break
                
    finally:
        del audio_src, pv, tempo_detector

In [11]:
# 放入你的音频文件路径
audio_file = "1_audio.wav"

stream = aubio_running_global_generator(audio_file)
    
print("🎵 正在流式播放/分析音频...")

final_global_report = None

for idx, data in enumerate(stream):
    # 模拟每隔 50 帧，在控制台刷新一次“截至目前”的全局统计数据
    if idx % 50 == 0:
        print(f"[{data['time']:.1f}s] 当前预估BPM: {data['running_global']['estimated_bpm']:.1f} | " 
                f"累计平均质心: {data['running_global']['average_spectral_centroid']:.1f} Hz | "
                f"当前瞬时能量: {data['local']['rms']:.4f}")
        
    # 当到达最后一帧时，把数据捕获下来作为最终报告
    if data['is_last_frame']:
        final_global_report = data['running_global']

# --------------------------------------------------
# 当生成器处理完整个音频时，这些数据也成为了最终结果
# --------------------------------------------------
print("\n" + "="*20 + " 🥁 最终全曲分析报告 🥁 " + "="*20)
print(f"🎵 最终全曲节奏 (BPM): {final_global_report['estimated_bpm']:.2f}")
print(f"☀️ 最终整体明亮度 (Overall Brightness): {final_global_report['overall_brightness_hz']:.1f} Hz")
print(f"📉 最终全曲总动态范围 (Dynamic Range): {final_global_report['dynamic_range']:.5f}")

🎵 正在流式播放/分析音频...
[0.0s] 当前预估BPM: 0.0 | 累计平均质心: 0.0 Hz | 当前瞬时能量: 0.0000
[0.6s] 当前预估BPM: 0.0 | 累计平均质心: 39.5 Hz | 当前瞬时能量: 0.0445
[1.2s] 当前预估BPM: 0.0 | 累计平均质心: 47.8 Hz | 当前瞬时能量: 0.1561
[1.7s] 当前预估BPM: 121.3 | 累计平均质心: 49.2 Hz | 当前瞬时能量: 0.1053
[2.3s] 当前预估BPM: 121.3 | 累计平均质心: 48.5 Hz | 当前瞬时能量: 0.0965
[2.9s] 当前预估BPM: 121.3 | 累计平均质心: 49.1 Hz | 当前瞬时能量: 0.0056
[3.5s] 当前预估BPM: 96.2 | 累计平均质心: 48.3 Hz | 当前瞬时能量: 0.2776
[4.1s] 当前预估BPM: 96.2 | 累计平均质心: 48.6 Hz | 当前瞬时能量: 0.1247
[4.6s] 当前预估BPM: 98.6 | 累计平均质心: 46.7 Hz | 当前瞬时能量: 0.1009
[5.2s] 当前预估BPM: 98.6 | 累计平均质心: 47.1 Hz | 当前瞬时能量: 0.0709
[5.8s] 当前预估BPM: 98.6 | 累计平均质心: 46.5 Hz | 当前瞬时能量: 0.0825
[6.4s] 当前预估BPM: 98.5 | 累计平均质心: 47.8 Hz | 当前瞬时能量: 0.0154
[7.0s] 当前预估BPM: 98.5 | 累计平均质心: 47.4 Hz | 当前瞬时能量: 0.1902
[7.5s] 当前预估BPM: 120.6 | 累计平均质心: 48.7 Hz | 当前瞬时能量: 0.0241
[8.1s] 当前预估BPM: 120.6 | 累计平均质心: 49.5 Hz | 当前瞬时能量: 0.0396
[8.7s] 当前预估BPM: 120.6 | 累计平均质心: 49.3 Hz | 当前瞬时能量: 0.1504
[9.3s] 当前预估BPM: 120.8 | 累计平均质心: 49.1 Hz | 当前瞬时能量: 0.0664
[9.9s] 当前预估BPM: 120.8 | 累计平均

In [16]:
from transformers import pipeline

# Initialize an audio classification pipeline using a fine-tuned model
classifier = pipeline("audio-classification", model="dima806/music_genres_classification", trust_remote_code=True)

# Extract genres directly from the file
genres = classifier("1_audio.wav")

# find max
max_genre = max(genres, key=lambda x: x['score'])
print(max_genre['label'], max_genre['score'])

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

hiphop 0.9259333610534668
